# Sıfırdan Dil Modeli · Bölüm 1
## Metinden sayılara: ilk tokenizer'ımız

Bu defterde bir dil modelinin ilk parçasını yazıyoruz: metni sayılara, sayıları yeniden metne çeviren **tokenizer**.

Adımlar Sebastian Raschka'nın *Build a Large Language Model (From Scratch)* kitabının 2. bölümünü izler. Kod, kitabın açık kaynak deposundaki kodun ([rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Apache-2.0) Türkçe metin için uyarlamasıdır.

Hücreleri sırayla çalıştırın: hücreye tıklayın, **Shift + Enter**'a basın.

## 1. İlk hücre
Bu kutulara **hücre** diyoruz. İçine yazdığımız Python kodu, Google'ın bulutundaki bir bilgisayarda çalışır.

In [ ]:
print("Merhaba, dil modeli!")

## 2. Veriyi indirmek
Modelimizin okuyacağı ilk metin Ömer Seyfettin'in **Kaşağı** öyküsü (yazarı 1920'de öldü; metin kamu malı). Dosya serinin GitHub deposunda duruyor; aşağıdaki hücre onu Colab'a indirir. Dosya zaten varsa yeniden indirmez.

In [ ]:
import os
import urllib.request

file_path = "kasagi.txt"
url = ("https://raw.githubusercontent.com/kocahmet1/sifirdan-dil-modeli/"
       "main/veri/kasagi.txt")

if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)

Metni okuyup kaç karakter olduğuna ve ilk 99 karakterine bakalım. `encoding="utf-8"` Türkçe harfler (ç, ğ, ı, ö, ş, ü) için şart.

In [ ]:
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Toplam karakter sayısı:", len(raw_text))
print(raw_text[:99])

## 3. Metni parçalara bölmek
Önce kısa bir cümlede deneyelim. `re`, Python'un düzenli ifade (regex) kütüphanesi; `re.split` metni verdiğimiz desene göre böler. `\s` "boşluk" demek. Deseni parantez içine aldığımız için boşluklar da listede kalır.

In [ ]:
import re

text = "Merhaba, dünya. Bu bir deneme."
result = re.split(r'(\s)', text)
print(result)

Virgül ve nokta kelimelere yapışık kaldı. Desene onları da ekleyelim: `[,.]` "virgül ya da nokta", `|` "ya da" demek.

In [ ]:
result = re.split(r'([,.]|\s)', text)
print(result)

Listede boş dizgiler (`''`) ve boşluklar (`' '`) kaldı. `item.strip()` baştaki ve sondaki boşlukları kırpar; kırpınca boş kalan öğeleri listeden atıyoruz.

In [ ]:
result = [item for item in result if item.strip()]
print(result)

Gerçek metinde başka işaretler de var: soru ve ünlem işareti, tırnak, kesme işareti, `--`, konuşma çizgisi (—) ve üç nokta (…). Hepsini desene ekliyoruz. Türkçede kesme işareti özel ismi ekinden ayırır; bakın ne oluyor:

In [ ]:
text = "Merhaba, dünya! Ankara'ya -- gerçekten -- gittin mi?"
result = re.split(r'([,.:;?_!"()\'…—]|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

Aynı iki satırı bütün öyküye uyguluyoruz.

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\'…—]|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))
print(preprocessed[:30])

## 4. Sözlük: her parçaya bir numara
`set` tekrarları atar, `sorted` sıralar. Geriye her farklı parçadan bir tane kalır.

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

`enumerate` öğeleri 0, 1, 2… numaralarıyla birlikte verir. Bununla "parça → numara" sözlüğünü (Python'da `dict`) kuruyor, ilk 21 girdiye bakıyoruz.

In [ ]:
vocab = {token: integer for integer, token in enumerate(all_words)}

for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 20:
        break

Sıralama tuhaf görünebilir: Python harfleri Unicode numaralarına göre sıralar, bu yüzden Ç, Ş, ı gibi harflerle başlayan kelimeler sona düşer. Bizim için önemli değil; tek amaç her parçaya sabit bir numara vermek. Sözlüğün son girdileri:

In [ ]:
for item in list(vocab.items())[-8:]:
    print(item)

## 5. İlk tokenizer'ımız
Hepsini bir **sınıfta** topluyoruz. `encode` metni numaralara, `decode` numaraları yeniden metne çevirir.

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\'…—]|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'…])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Gümüş gibi parlayan bu güzel kaşağının dişlerine baktım."
ids = tokenizer.encode(text)
print(ids)

In [ ]:
print(tokenizer.decode(ids))

## 6. Sözlükte olmayan kelimeler
Tokenizer'ımız yalnızca öyküde geçen parçaları tanıyor. Öyküde hiç geçmeyen bir kelime verirsek ne olur?

In [ ]:
text = "Merhaba, nasılsın?"
print(tokenizer.encode(text))

Sözlüğe iki özel parça ekliyoruz: `<|unk|>` (*unknown*: bilinmeyen parça) ve `<|endoftext|>` (metnin sonu: art arda gelen iki ayrı metni birbirinden ayırır).

In [ ]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token: integer for integer, token in enumerate(all_tokens)}
print(len(vocab))

In [ ]:
for item in list(vocab.items())[-5:]:
    print(item)

İkinci sürüm, sözlükte olmayan her parçayı `<|unk|>` ile değiştiriyor. Birinci sürümden tek farkı `encode`'daki bir satır.

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\'…—]|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else "<|unk|>"
                        for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'…])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Merhaba, nasılsın?"
text2 = "Hasan'la Dadaruh dere kenarına inmişlerdi."
text = " <|endoftext|> ".join((text1, text2))
print(text)

In [ ]:
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

## 7. Türkçenin sorunu
Türkçede bir köke ekler eklenerek yüzlerce kelime türetilir. Kelime düzeyindeki tokenizer'ımız her birini ayrı, bağımsız bir kelime sanıyor:

In [ ]:
words = ["ev", "evler", "evlerimiz", "evlerimizde",
         "evlerimizdeki", "evlerimizdekiler", "evlerimizdekilerden"]

for word in words:
    print(f"{word:20} → {tokenizer.encode(word)}")

Öyküde aynı kökten kaç farklı kelime var? `Counter` her parçanın kaç kez geçtiğini sayar.

In [ ]:
from collections import Counter

counts = Counter(preprocessed)
print(sorted(w for w in counts if w.lower().startswith("kaşağı")))

In [ ]:
once = [w for w, c in counts.items() if c == 1]
print("Farklı parça sayısı:", len(counts))
print("Yalnızca bir kez geçen:", len(once))

## Özet
Bugünkü tokenizer metni sayılara çevirip geri getirebiliyor ama iki zayıf noktası var: sözlükte olmayan her kelime `<|unk|>` oluyor ve Türkçenin ekleri sözlüğü şişiriyor. **Bölüm 2'de** kelimeleri değil kelime *parçalarını* sayan yöntemi, BPE'yi, sıfırdan yazacağız.

**Kendin dene**
- `text` değişkenine kendi cümlenizi yazın, `encode` ve `decode` edin.
- Desenden `…—` işaretlerini silip hücreleri yeniden çalıştırın: parça sayısı nasıl değişti?

---
Kaynak: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Manning 2024, bölüm 2. Kod: [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (Apache-2.0), Türkçe uyarlama: Sıfırdan Dil Modeli.